# RAG with task objects

Load the complete bank produced by `06.test.rag.object.question_with_tasks.ipynb`. Retrieve whole task objects from the existing **task_embeddings** collection and answer their questions with cited procedures, parameters, or documented code.

Embeddings use **172.27.74.49:11434**; generation uses **172.27.74.16:11434**. Expected task IDs and reference excerpts are used only for evaluation. Retrieval receives the question and category. This notebook opens an existing index and reports missing coverage; it does not build embeddings or alter summaries.

In [ ]:
# | notest
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").is_file() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
if not (PROJ_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set PROJ_ROOT to the ribosome project directory")
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
load_dotenv(PROJ_ROOT / ".env", override=False)
from ribosome.ollama_config import OLLAMA_EMBED_HOST, OLLAMA_RAG_HOST, OLLAMA_TRUST_ENV

RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
REQUEST_TIMEOUT = 300
BANK_PATH = PROJ_ROOT / "db/task_rag_tests/question_with_tasks/bank.json"
HEADERS = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]} if os.getenv("OLLAMA_API_KEY") else {}

import math
from urllib.parse import urlsplit
from utils.task_rag_questions import load_task_bank, select_cases
from utils.task_rag import TaskRAG, TaskEvaluationRun, task_index_coverage, require_task_coverage, rerank_tasks
from utils.object_rag_evaluation import QueryEmbeddingCache, cache_readonly_snapshots, grouped_metrics
from utils.object_rag_category_questions import CategoryPromptGenerator, CATEGORY_SYSTEM_PROMPT

EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")
MODEL_DIRECTORY = EMBED_MODEL.replace("/", "_").replace(":", "-")
HOST_DIRECTORY = urlsplit(OLLAMA_EMBED_HOST).netloc.replace(":", "-")
INDEX_DIR = Path(os.getenv("TASK_RAG_INDEX") or f"db/task_embeddings/{MODEL_DIRECTORY}-{HOST_DIRECTORY}")
if not INDEX_DIR.is_absolute():
    INDEX_DIR = PROJ_ROOT / INDEX_DIR
HOSTS = [OLLAMA_EMBED_HOST]
COLLECTION_NAME = "task_embeddings"
TOP_K = 8
CONCURRENCY = 2
MAX_CONTEXT_CHARS = 24000
MAX_OUTPUT_TOKENS = 2048
CATEGORY_SCORE_BOOST = float(os.getenv("TASK_RAG_CATEGORY_BOOST", "0.15"))
if not math.isfinite(CATEGORY_SCORE_BOOST) or CATEGORY_SCORE_BOOST < 0:
    raise ValueError("TASK_RAG_CATEGORY_BOOST must be finite and nonnegative")
SELECTED_CASE_IDS = None  # None runs all questions; otherwise supply known case IDs.
ANSWER_PREVIEW_LIMIT = 10

def rerank_by_category(hits, category):
    return rerank_tasks(hits, category, CATEGORY_SCORE_BOOST)


## Load questions and select cases

In [ ]:
# | notest
bank = load_task_bank(BANK_PATH, PROJ_ROOT)
cases = select_cases(bank["cases"], SELECTED_CASE_IDS)
display(pd.DataFrame(bank["documents"]))
display(pd.DataFrame([{key: case[key] for key in ("id", "category", "profile", "task_id", "question")}
                      for case in cases[:20]]))
print(f"Selected {len(cases)} of {len(bank['cases'])} task questions")


## Open the task index and check selected-task coverage

The collection also contains ordinary objects, so the retriever explicitly admits only `type == "task"`. Coverage distinguishes missing documents, missing tasks, missing vectors and stale content. Complete or refresh the task index with the embedding notebook if any selected task is not ready.

In [ ]:
# | notest
import chromadb
from ollama import AsyncClient
if not (INDEX_DIR / "catalog.sqlite3").is_file() or not (INDEX_DIR / "chroma").is_dir():
    raise FileNotFoundError(f"Existing task index required: {INDEX_DIR}")
if not RAG_MODEL:
    raise ValueError("Configure OLLAMA_RAG_MODEL or OLLAMA_SUMMARY_MODEL")
embedding_client = AsyncClient(host=OLLAMA_EMBED_HOST, headers=HEADERS,
                              timeout=REQUEST_TIMEOUT, trust_env=OLLAMA_TRUST_ENV)
generation_client = AsyncClient(host=OLLAMA_RAG_HOST, headers=HEADERS,
                               timeout=REQUEST_TIMEOUT, trust_env=OLLAMA_TRUST_ENV)
app = None
try:
    collection = chromadb.PersistentClient(path=str(INDEX_DIR / "chroma")).get_collection(
        COLLECTION_NAME, embedding_function=None)
    app = TaskRAG(INDEX_DIR, collection, embedding_client, CategoryPromptGenerator(generation_client),
                  mapping_dir=PROJ_ROOT / "proj_config/coding_categories", embed_model=EMBED_MODEL,
                  embedding_config=dict(provider="ollama-pool", hosts=HOSTS), rag_model=RAG_MODEL,
                  max_context_chars=MAX_CONTEXT_CHARS, max_output_tokens=MAX_OUTPUT_TOKENS,
                  rerank=rerank_by_category)
    cache_readonly_snapshots(app)
    coverage = task_index_coverage(app, cases)
    display(pd.DataFrame(coverage))
    require_task_coverage(coverage)
    query_cache = QueryEmbeddingCache(embedding_client, EMBED_MODEL, app.embedding_config,
                                     app.dimension, BANK_PATH.parent / "query_embeddings")
    app.embedder = query_cache
    evaluation = TaskEvaluationRun(app, cases, BANK_PATH.parent / "results", top_k=TOP_K,
        settings=dict(question_variant="task", rag_host=OLLAMA_RAG_HOST,
                      category_system_prompt=CATEGORY_SYSTEM_PROMPT,
                      category_boost=CATEGORY_SCORE_BOOST, collection=COLLECTION_NAME))
except BaseException:
    if app is not None:
        app.close()
    await embedding_client._client.aclose()
    await generation_client._client.aclose()
    raise


## Retrieve task evidence

Successful query embeddings and retrieval stages are reused. Expected-task metrics require the exact task in the correct source/profile; retrieving a different task from the same manual does not count. Context metrics also require the complete task source to reach the model.

In [ ]:
# | notest
await query_cache.prepare([case["question"] for case in cases])
retrieval_metrics = await evaluation.retrieve_cases(cases)
display(retrieval_metrics)


## Answer questions and review results

Answers use frozen retrieved evidence and may describe procedures or parameters without code. Invalid responses receive one corrective retry. Successful answers are reused and failed answers remain retryable. Results are saved atomically in an independent task-results directory.

In [ ]:
# | notest
metrics = await evaluation.run(cases, concurrency=CONCURRENCY)
display(metrics)
display(grouped_metrics(metrics, ["category", "profile"]))
evaluation.show_answers(cases[:ANSWER_PREVIEW_LIMIT])


## Inspect one case

In [ ]:
# | notest
INSPECT_CASE_ID = None  # Set a selected case ID to inspect saved evidence and reviewer references.
if INSPECT_CASE_ID is not None:
    evaluation.inspect(INSPECT_CASE_ID)


## Close clients

In [ ]:
# | notest
try:
    if app is not None:
        app.close()
finally:
    await embedding_client._client.aclose()
    await generation_client._client.aclose()
